# Chapter 15: Evaluating LLM Outputs

**Book:** [LLMs for Business & Quality Analysts](../index.html)

## Lab Overview
Build a **Validation Framework** that systematically evaluates LLM-generated BA/QA artifacts for accuracy, completeness, consistency, and quality. Learn LLM-as-judge patterns.

## Prerequisites
- Python 3.9+
- OpenAI API key
- Completed previous chapter labs


In [ ]:
# Setup
!pip install -q openai python-dotenv

import os
import json
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI()
MODEL = 'gpt-4o'

## 1. Define Evaluation Criteria

Create rubrics for evaluating different types of BA/QA outputs.


In [ ]:
# Evaluation rubrics for different artifact types
rubrics = {
    'user_story': {
        'criteria': [
            {'name': 'format_compliance', 'description': 'Follows As a/I want/So that format', 'weight': 0.15},
            {'name': 'invest_criteria', 'description': 'Meets INVEST (Independent, Negotiable, Valuable, Estimable, Small, Testable)', 'weight': 0.25},
            {'name': 'acceptance_criteria_quality', 'description': 'Has clear, testable Given/When/Then acceptance criteria', 'weight': 0.25},
            {'name': 'completeness', 'description': 'Includes all necessary details (priority, points, dependencies)', 'weight': 0.15},
            {'name': 'clarity', 'description': 'Clear, unambiguous language', 'weight': 0.20}
        ],
        'scale': '1-5 where 1=Poor, 2=Below Average, 3=Average, 4=Good, 5=Excellent'
    },
    'test_case': {
        'criteria': [
            {'name': 'coverage', 'description': 'Covers positive, negative, boundary, and edge cases', 'weight': 0.25},
            {'name': 'specificity', 'description': 'Steps are specific and reproducible', 'weight': 0.20},
            {'name': 'test_data', 'description': 'Includes concrete test data values', 'weight': 0.15},
            {'name': 'expected_results', 'description': 'Expected results are verifiable and specific', 'weight': 0.20},
            {'name': 'traceability', 'description': 'Clearly traces back to requirements', 'weight': 0.20}
        ],
        'scale': '1-5 where 1=Poor, 2=Below Average, 3=Average, 4=Good, 5=Excellent'
    }
}

print('Evaluation rubrics defined for:', list(rubrics.keys()))

## 2. LLM-as-Judge Evaluation

Use a separate LLM call to evaluate the quality of generated artifacts.


In [ ]:
def evaluate_artifact(artifact: str, artifact_type: str, context: str = '') -> dict:
    """Use LLM-as-judge to evaluate a BA/QA artifact."""
    rubric = rubrics[artifact_type]
    
    prompt = f"""You are a senior BA/QA quality reviewer. Evaluate this {artifact_type} artifact 
against the rubric below.

Artifact to evaluate:
{artifact}

{'Context: ' + context if context else ''}

Evaluation Rubric (scale: {rubric['scale']}):
{json.dumps(rubric['criteria'], indent=2)}

For each criterion, provide:
- score: 1-5
- justification: specific explanation with examples from the artifact
- improvement_suggestion: how to improve this aspect

Also provide:
- overall_score: weighted average (round to 1 decimal)
- strengths: top 2 strengths
- weaknesses: top 2 weaknesses
- verdict: PASS (overall >= 3.5) or NEEDS_REVISION (overall < 3.5)

Return as JSON. Return ONLY valid JSON."""
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.2
    )
    return json.loads(response.choices[0].message.content)

# Generate a user story and then evaluate it
gen_response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Write a user story with acceptance criteria for: email notification preferences in a project management tool. Include story points and priority.'}],
    temperature=0.5
)
generated_story = gen_response.choices[0].message.content
print('GENERATED ARTIFACT:')
print(generated_story)
print('\n' + '='*60 + '\n')

# Evaluate it
evaluation = evaluate_artifact(generated_story, 'user_story')
print('EVALUATION RESULTS:')
print(f"Overall Score: {evaluation['overall_score']} - {evaluation['verdict']}")
print(f"Strengths: {evaluation['strengths']}")
print(f"Weaknesses: {evaluation['weaknesses']}")

## 3. Consistency Checking

Compare multiple LLM outputs for the same prompt to measure consistency.


In [ ]:
# Generate the same artifact 3 times and measure consistency
prompt = 'Write 3 test cases for a shopping cart "Remove Item" feature.'
outputs = []
for i in range(3):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.5
    )
    outputs.append(response.choices[0].message.content)

# Use LLM to assess consistency
consistency_prompt = f"""Compare these 3 outputs generated from the same prompt and assess consistency.

Prompt: {prompt}

Output 1:
{outputs[0]}

Output 2:
{outputs[1]}

Output 3:
{outputs[2]}

Assess:
1. structural_consistency: Do they follow the same format? (1-5)
2. content_overlap: What percentage of test scenarios are shared across outputs?
3. quality_variance: How much does quality vary between outputs? (low/medium/high)
4. unique_insights: Valuable ideas that only appear in one output
5. recommendation: Should you use a single run, or merge the best of multiple runs?

Return as JSON. Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': consistency_prompt}],
    temperature=0.2
)
consistency = json.loads(response.choices[0].message.content)
print('CONSISTENCY ANALYSIS:')
print(json.dumps(consistency, indent=2))

In [ ]:
# Factual accuracy check: verify generated content against source requirements
source_requirement = """REQ-100: The system shall allow users to reset their password. 
The reset link shall expire after 24 hours. Maximum 3 reset attempts per day. 
The new password must not match any of the last 5 passwords."""

generated_tests = """TC-001: Verify password reset link expires after 12 hours
TC-002: Verify user can request up to 5 password resets per day
TC-003: Verify new password cannot match last 3 passwords
TC-004: Verify reset email is sent within 2 minutes"""

accuracy_prompt = f"""Check the factual accuracy of these generated test cases against the source requirement.

Source Requirement:
{source_requirement}

Generated Test Cases:
{generated_tests}

For each test case, identify:
- is_accurate: true/false
- discrepancy: what's wrong (if any)
- source_value: what the requirement actually says
- generated_value: what the test case says

Return JSON with: test_cases (array), accuracy_score (percentage), hallucinations_found (count).
Return ONLY valid JSON."""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': accuracy_prompt}],
    temperature=0.1
)
accuracy = json.loads(response.choices[0].message.content)
print(f"Accuracy Score: {accuracy['accuracy_score']}%")
print(f"Hallucinations Found: {accuracy['hallucinations_found']}")
for tc in accuracy['test_cases']:
    status = 'ACCURATE' if tc['is_accurate'] else 'INACCURATE'
    print(f"  [{status}] {tc.get('test_case_id', tc.get('tc_id', 'N/A'))}: {tc.get('discrepancy', 'None')}")

## Exercises
1. Build an automated evaluation pipeline that scores all artifacts generated in previous chapters.
2. Create a benchmark dataset of human-written artifacts and compare LLM output quality.
3. Implement an iterative improvement loop: generate, evaluate, improve, re-evaluate.
4. Build a dashboard that tracks quality metrics over time as you refine your prompts.
